# The idea is to create a representative 1-billion-token sample from the Australia dataset, but store the final result in one single parquet file.

Instead of just randomly picking a few large files until we reach 1 billion tokens, we first scan all the Australia parquet files and only read the token_count column. This gives us a manifest showing how many tokens each file, row group, Common Crawl dump, and year contains.

Then we divide the 1-billion-token target across the different Common Crawl dumps according to their original token proportions. For example, if one dump contains about 3% of the full Australia dataset, then it should contribute about 3% of the 1B-token sample. This helps avoid taking data from only one year or only a few source files.

Within each dump, we randomly sample row groups, and if the last row group is too large, we randomly sample rows inside that row group. This keeps the sample random but still spread across many dumps, years, and files.

For each selected chunk, we keep the original columns such as text, url, date, language, language_score, and token_count. We also add extra metadata columns such as country, source_file, source_dump, and source_year, so we can later explain where the sample came from.

Finally, instead of writing many small parquet shards, we use a ParquetWriter to append all selected chunks into one compressed parquet file. So the output is still representative and randomly sampled across the dataset, but it is delivered as a single Australia_1B_representative.parquet file.

In [16]:
%pip install -U huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.5/661.5 kB 13.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 21.9 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12/12 [huggingface_hub] [huggingface_hub]
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 55.9 MB/s  0:00:00 eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [3]:
from pathlib import Path
import json
import random
import re

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc

# =========================
# Config
# =========================

BASE_DIR = Path("/home/jovyan/data")
COUNTRY = "Australia"

INPUT_DIR = BASE_DIR / COUNTRY
OUTPUT_DIR = BASE_DIR / "sampled_1B_single"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = OUTPUT_DIR / "Australia_1B_representative.parquet"
SUMMARY_FILE = OUTPUT_DIR / "Australia_1B_representative_summary.json"

TARGET_TOKENS = 1_000_000_000
RANDOM_SEED = 42

random.seed(RANDOM_SEED)

if OUTPUT_FILE.exists():
    raise FileExistsError(
        f"Output file already exists:\n{OUTPUT_FILE}\n"
        "Please delete it or choose another output filename before rerunning."
    )


# =========================
# Helper functions
# =========================

def parse_dump_and_year(file_path, country):
    """
    Expected path pattern:
    /home/jovyan/data/Australia/CC-MAIN-2022-33/chunk_0.parquet
    """
    parts = file_path.parts
    country_index = parts.index(country)
    dump = parts[country_index + 1]

    year = ""
    if dump.startswith("CC-MAIN-"):
        dump_parts = dump.split("-")
        if len(dump_parts) >= 3:
            year = dump_parts[2]

    return dump, year


def cast_string_to_large_string(table):
    """
    Avoid ArrowInvalid: offset overflow for large text columns.
    """
    new_columns = []
    new_fields = []

    for field, column in zip(table.schema, table.columns):
        if pa.types.is_string(field.type):
            new_columns.append(column.cast(pa.large_string()))
            new_fields.append(pa.field(field.name, pa.large_string()))
        else:
            new_columns.append(column)
            new_fields.append(field)

    return pa.Table.from_arrays(new_columns, schema=pa.schema(new_fields))


def append_metadata_columns(table, country, source_file, source_dump, source_year):
    """
    Add metadata columns for dataset card / paper / later analysis.
    """
    n = table.num_rows

    metadata = {
        "country": country,
        "source_file": str(source_file),
        "source_dump": source_dump,
        "source_year": source_year,
    }

    for col_name, value in metadata.items():
        if col_name not in table.column_names:
            table = table.append_column(
                col_name,
                pa.array([value] * n, type=pa.large_string())
            )

    return table


def calculate_dump_quotas(manifest_df, target_tokens):
    """
    Allocate sampling quota to each dump proportionally to its size.
    """
    dump_tokens = manifest_df.groupby("dump")["tokens"].sum().to_dict()
    total_tokens = sum(dump_tokens.values())

    quotas = {}
    for dump, tokens in dump_tokens.items():
        quotas[dump] = int(target_tokens * tokens / total_tokens)

    # Fix rounding error.
    diff = target_tokens - sum(quotas.values())
    if diff != 0:
        largest_dump = max(dump_tokens, key=dump_tokens.get)
        quotas[largest_dump] += diff

    return quotas


def sample_partial_row_group(file_path, row_group, remaining_tokens):
    """
    Randomly sample rows inside one row group until reaching remaining_tokens.
    """
    pf = pq.ParquetFile(file_path)

    token_table = pf.read_row_group(row_group, columns=["token_count"])
    token_counts = token_table["token_count"].to_pylist()

    indices = list(range(len(token_counts)))
    random.shuffle(indices)

    selected_indices = []
    selected_tokens = 0

    for i in indices:
        if selected_tokens >= remaining_tokens:
            break

        token_count = token_counts[i]
        if token_count is None:
            continue

        selected_indices.append(i)
        selected_tokens += int(token_count)

    if not selected_indices:
        return None, 0

    full_table = pf.read_row_group(row_group)
    full_table = cast_string_to_large_string(full_table)

    selected_table = full_table.take(pa.array(selected_indices, type=pa.int64()))

    return selected_table, selected_tokens


# =========================
# Step 1: Build Australia manifest
# =========================

print("=" * 100)
print("Building manifest for Australia...")

parquet_files = list(INPUT_DIR.rglob("*.parquet"))

print("Input dir:", INPUT_DIR)
print("Number of input parquet files:", len(parquet_files))

manifest = []

for file_idx, file_path in enumerate(parquet_files):
    pf = pq.ParquetFile(file_path)
    dump, year = parse_dump_and_year(file_path, COUNTRY)

    for rg_idx in range(pf.num_row_groups):
        token_table = pf.read_row_group(rg_idx, columns=["token_count"])
        token_sum = pc.sum(token_table["token_count"]).as_py()

        if token_sum is None:
            token_sum = 0

        manifest.append({
            "country": COUNTRY,
            "file_path": str(file_path),
            "dump": dump,
            "year": year,
            "row_group": rg_idx,
            "rows": token_table.num_rows,
            "tokens": int(token_sum),
        })

    if (file_idx + 1) % 20 == 0:
        print(f"Scanned {file_idx + 1}/{len(parquet_files)} files")

manifest_df = pd.DataFrame(manifest)

print("Manifest entries:", len(manifest_df))
print("Total Australia tokens in source:", f"{manifest_df['tokens'].sum():,}")
print("Dumps:", manifest_df["dump"].nunique())
print("Years:", manifest_df["year"].nunique())


# =========================
# Step 2: Representative sampling into one parquet
# =========================

print("=" * 100)
print("Sampling Australia into ONE parquet file...")

dump_quotas = calculate_dump_quotas(manifest_df, TARGET_TOKENS)

writer = None

total_actual_tokens = 0
total_rows = 0
tables_written = 0

selected_records = []

used_source_files = set()
used_dumps = set()
used_years = set()

try:
    for dump, dump_quota in sorted(dump_quotas.items()):
        if total_actual_tokens >= TARGET_TOKENS:
            break

        dump_df = manifest_df[manifest_df["dump"] == dump].copy()
        dump_entries = dump_df.to_dict("records")
        random.shuffle(dump_entries)

        dump_actual_tokens = 0
        dump_rows = 0
        dump_written = 0

        print("-" * 100)
        print("Dump:", dump)
        print("Dump quota:", f"{dump_quota:,}")
        print("Candidate row groups:", len(dump_entries))

        for entry in dump_entries:
            if total_actual_tokens >= TARGET_TOKENS:
                break

            if dump_actual_tokens >= dump_quota:
                break

            file_path = entry["file_path"]
            row_group = int(entry["row_group"])
            rg_tokens = int(entry["tokens"])
            year = entry["year"]

            remaining_country_tokens = TARGET_TOKENS - total_actual_tokens
            remaining_dump_tokens = dump_quota - dump_actual_tokens
            remaining_tokens = min(remaining_country_tokens, remaining_dump_tokens)

            pf = pq.ParquetFile(file_path)

            # Include the whole row group if it fits the quota.
            if rg_tokens <= remaining_tokens:
                table = pf.read_row_group(row_group)
                table = cast_string_to_large_string(table)
                selected_tokens = rg_tokens

            # Otherwise sample rows inside this row group.
            else:
                table, selected_tokens = sample_partial_row_group(
                    file_path=file_path,
                    row_group=row_group,
                    remaining_tokens=remaining_tokens
                )

                if table is None:
                    continue

            table = append_metadata_columns(
                table=table,
                country=COUNTRY,
                source_file=file_path,
                source_dump=dump,
                source_year=year,
            )

            # Create the single parquet writer using the first table's schema.
            if writer is None:
                writer = pq.ParquetWriter(
                    OUTPUT_FILE,
                    table.schema,
                    compression="zstd"
                )

            writer.write_table(table)

            rows_added = table.num_rows

            total_actual_tokens += selected_tokens
            dump_actual_tokens += selected_tokens

            total_rows += rows_added
            dump_rows += rows_added

            tables_written += 1
            dump_written += 1

            used_source_files.add(file_path)
            used_dumps.add(dump)
            used_years.add(year)

            selected_records.append({
                "country": COUNTRY,
                "dump": dump,
                "year": year,
                "source_file": file_path,
                "row_group": row_group,
                "tokens": selected_tokens,
                "rows": rows_added,
            })

        print(
            f"Finished dump {dump} | "
            f"Dump tokens: {dump_actual_tokens:,} | "
            f"Total tokens: {total_actual_tokens:,} | "
            f"Rows: {total_rows:,} | "
            f"Tables written into single parquet: {tables_written}"
        )

finally:
    if writer is not None:
        writer.close()


# =========================
# Step 3: Save summary
# =========================

summary = {
    "country": COUNTRY,
    "target_tokens": TARGET_TOKENS,
    "actual_tokens": total_actual_tokens,
    "rows": total_rows,
    "output_parquet_files": 1,
    "tables_written_into_single_parquet": tables_written,
    "source_files_used": len(used_source_files),
    "dumps_used": len(used_dumps),
    "years_used": len([y for y in used_years if y]),
    "output_file": str(OUTPUT_FILE),
    "output_size_gb": OUTPUT_FILE.stat().st_size / 1024**3,
}

with open(SUMMARY_FILE, "w", encoding="utf-8") as f:
    json.dump(
        {
            "summary": summary,
            "selected_sources": selected_records,
        },
        f,
        indent=2
    )

print("=" * 100)
print("Australia single-parquet sampling finished.")
print(json.dumps(summary, indent=2))

Building manifest for Australia...
Input dir: /home/jovyan/data/Australia
Number of input parquet files: 455
Scanned 20/455 files
Scanned 40/455 files
Scanned 60/455 files
Scanned 80/455 files
Scanned 100/455 files
Scanned 120/455 files
Scanned 140/455 files
Scanned 160/455 files
Scanned 180/455 files
Scanned 200/455 files
Scanned 220/455 files
Scanned 240/455 files
Scanned 260/455 files
Scanned 280/455 files
Scanned 300/455 files
Scanned 320/455 files
Scanned 340/455 files
Scanned 360/455 files
Scanned 380/455 files
Scanned 400/455 files
Scanned 420/455 files
Scanned 440/455 files
Manifest entries: 777
Total Australia tokens in source: 294,183,336,933
Dumps: 109
Years: 13
Sampling Australia into ONE parquet file...
----------------------------------------------------------------------------------------------------
Dump: CC-MAIN-2013-20
Dump quota: 6,369,773
Candidate row groups: 5
Finished dump CC-MAIN-2013-20 | Dump tokens: 6,369,823 | Total tokens: 6,369,823 | Rows: 9,286 | Tables w

In [5]:
from pathlib import Path
import pyarrow.parquet as pq
import pyarrow.compute as pc
import pandas as pd

output_file = Path("/home/jovyan/data/sampled_1B_single/Australia_1B_representative.parquet")

print("Output file exists:", output_file.exists())
print("Output file:", output_file)
print("Size GB:", f"{output_file.stat().st_size / 1024**3:.2f}")

pf = pq.ParquetFile(output_file)

print("Number of parquet files: 1")
print("Rows from metadata:", f"{pf.metadata.num_rows:,}")
print("Row groups:", pf.num_row_groups)
print("Columns:")
print(pf.schema.names)

table = pq.read_table(
    output_file,
    columns=["token_count", "source_file", "source_dump", "source_year"]
)

total_tokens = pc.sum(table["token_count"]).as_py()
if total_tokens is None:
    total_tokens = 0

source_files_used = len(set(table["source_file"].to_pylist()))
dumps_used = len(set(table["source_dump"].to_pylist()))
years_used = len(set(y for y in table["source_year"].to_pylist() if y))

verify = {
    "tokens": int(total_tokens),
    "rows": table.num_rows,
    "source_files_used": source_files_used,
    "dumps_used": dumps_used,
    "years_used": years_used,
    "size_gb": output_file.stat().st_size / 1024**3,
}

pd.DataFrame([verify])

Output file exists: True
Output file: /home/jovyan/data/sampled_1B_single/Australia_1B_representative.parquet
Size GB: 1.96
Number of parquet files: 1
Rows from metadata: 1,670,584
Row groups: 109
Columns:
['text', 'id', 'dump', 'url', 'date', 'file_path', 'language', 'language_score', 'token_count', 'index', 'country', 'source_file', 'source_dump', 'source_year']


,tokens,rows,source_files_used,dumps_used,years_used,size_gb
0,1000000528,1670584,109,109,13,1.956864


In [6]:
from pathlib import Path

output_dir = Path("/home/jovyan/data/sampled_1B_single")

parquet_files = list(output_dir.rglob("*.parquet"))

print("Parquet files:")
for f in parquet_files:
    print(f)

print("Number of parquet files:", len(parquet_files))

Parquet files:
/home/jovyan/data/sampled_1B_single/Australia_1B_representative.parquet
Number of parquet files: 1
